# Zero-shot transfer of implied-volatility forecasts: the whole pipeline on Colab
Runs every stage of the paper with the data on Google Drive. Each stage is resumable: rerun a cell after a disconnect and it continues.

| Stage | Runtime | Time |
|---|---|---|
| market scripts | CPU | 1-3 h per market (downloads) |
| surfaces | CPU with many cores | hours (US longest) |
| states | CPU | minutes |
| train, initial | GPU | hours |
| paper, trading, figures | CPU | 2-3 h |

To re-score the published models of the original run instead, set `LAYOUT = 'drive'` and `ROOT` to the original `pm_universe` folder,
then run only the setup cells and the `paper`, `trading` and `figures` cells (outputs go to new folders; nothing is overwritten).

In [ ]:
# ---- setup ----
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/iv-transfer'):
    subprocess.run(['git', 'clone', 'https://github.com/alexmilekhin/iv-transfer', '/content/iv-transfer'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', '/content/iv-transfer[markets]'], check=True)
sys.path.insert(0, '/content/iv-transfer/src')

In [ ]:
# ---- settings ----
ROOT = '/content/drive/MyDrive/iv-transfer-data'   # data root on Drive (created if missing)
LAYOUT = 'standard'                                 # 'drive' = the folder layout of the original run (ROOT = .../pm_universe)
import os; os.makedirs(ROOT, exist_ok=True)
from ivtransfer.config import Layout, DriveLayout
from ivtransfer import pipeline as P
L = DriveLayout(ROOT) if LAYOUT == 'drive' else Layout(ROOT)
print(type(L).__name__, ROOT)

In [ ]:
# ---- 1. market data (CPU; downloads; run each once). The US script needs the dolt CLI, installed by the script itself. ----
for script in ['us', 'br', 'india', 'hongkong', 'canada']:
    subprocess.run([sys.executable, f'/content/iv-transfer/markets/{script}.py', '--root', ROOT], check=True)

In [ ]:
# ---- 2. universes and surface fits (CPU, as many cores as possible; resumable per underlying-year) ----
P.stage_universe(L)
P.stage_surfaces(L)

In [ ]:
# ---- 3. model-ready samples per market (CPU; minutes) ----
P.stage_states(L)

In [ ]:
# ---- 4. training (GPU runtime): final network, 10 seeds; initial network with curves and input importance ----
P.stage_train(L, n_seeds=10)
P.stage_initial(L)

In [ ]:
# ---- 5. every table and number, Section 8, Figure 2 (CPU) ----
P.stage_paper(L, n_seeds=10)
P.stage_trading(L, n_seeds=10)
P.stage_figures(L.results_dir('paper'))
print('results in', L.results_dir('paper'))